# Notebook 1 – Data Preprocessing Fundamentals




## 1. What is Data Preprocessing?
Data preprocessing is the process of inspecting and transforming raw data into a reliable format for analysis or machine learning. It can include correcting data types, handling missing values, resolving duplicates, treating invalid values, encoding categories, scaling numeric features, and preparing train/validation/test datasets.

## 2. Why is preprocessing required?
- Real-world data can be incomplete, inconsistent, duplicated, noisy, or incorrectly typed.
- Many algorithms require numeric inputs and may be sensitive to scale.
- Poor-quality data can produce misleading analysis and unreliable predictions.
- A consistent pipeline makes training and future predictions use the same transformations.

## 3. Raw data vs clean data
| Raw data example | Possible issue | Cleaned approach |
|---|---|---|
| `" 25 "` | Numeric value stored as text with spaces | Trim and convert to number |
| `-5` for age | Invalid domain value | Investigate and correct/mark missing |
| `"Chennai"`, `"chennai"` | Inconsistent category spelling | Standardize using documented rules |
| Empty salary | Missing value | Investigate missingness and choose a suitable strategy |
| Same transaction repeated | Duplicate record | Verify business key before removing |

“Clean” does not mean changing every unusual value. Keep valid rare events and document every decision.

## 4. Data quality dimensions
- **Quality:** Overall fitness of data for its intended use.
- **Consistency:** Values follow the same definitions and formats across records/systems.
- **Completeness:** Required values are present.
- **Accuracy:** Values correctly represent the real-world fact.
- **Validity:** Values obey allowed types, ranges, formats, and business rules.
- **Integrity:** Relationships and constraints remain correct (for example, each order references an existing customer).

A value can be valid but inaccurate: an age of 35 is in a valid range but may still be wrong for that person.

## 5. Data leakage
Data leakage occurs when training uses information that would not genuinely be available at prediction time, making evaluation unrealistically optimistic.

Examples:
- Scaling the entire dataset before splitting it.
- Imputing missing values using statistics calculated from all rows.
- Including a post-outcome field such as `refund_issued` when predicting whether a refund will be requested.
- Allowing future records to inform a time-series prediction.

**Rule:** Split first. Fit preprocessing steps only on training data, then use the fitted transformations to transform validation, test, and future data.

## 6. Training, validation, and test data
- **Training set:** Used to fit model parameters and preprocessing statistics.
- **Validation set:** Used to compare models and tune hyperparameters.
- **Test set:** Held back for a final, relatively unbiased evaluation. Do not repeatedly tune against it.

Typical split examples are 70/15/15 or 80/10/10, but the best ratio depends on dataset size and task. For imbalanced classification, consider stratified splitting. For time-dependent data, split chronologically. For repeated people/devices, consider group-based splitting to prevent related records crossing splits.

## 7. Example dataset and initial checks

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [25, 31, np.nan, 42, 31],
    "city": ["Chennai", "chennai", "Salem", None, "Chennai"],
    "income": ["45000", "52000", "unknown", "70000", "52000"],
    "purchased": [0, 1, 0, 1, 1]
})

display(df)
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nSummary:")
display(df.describe(include="all").T)


split first, then preprocess
=
The following example uses a scikit-learn Pipeline and ColumnTransformer. The imputer, scaler, and encoder are fitted only on the training split.



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

sample = pd.DataFrame({
    "age": [22, 35, np.nan, 48, 29, 41, 37, np.nan, 52, 26, 33, 45],
    "city": ["Chennai", "Salem", "Chennai", None, "Madurai", "Salem",
             "Chennai", "Madurai", "Salem", "Chennai", None, "Madurai"],
    "income": [25000, 52000, 33000, 71000, np.nan, 65000,
               59000, 42000, 82000, 28000, 47000, 68000],
    "purchased": [0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1]
})

X = sample[["age", "city", "income"]]
y = sample["purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

numeric_features = ["age", "income"]
categorical_features = ["city"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
print("Test accuracy:", model.score(X_test, y_test))